# DatasetFilterResolver Tutorial

This tutorial demonstrates how to use the `DatasetFilterResolver` to filter samples across heterogeneous datasets using external YAML configuration.

## Overview

The `DatasetFilterResolver` solves a common problem: filtering experimental data across multiple datasets where experimental conditions are structured differently. Instead of hardcoding dataset-specific logic, you specify:

1. **Filter criteria**: Which values are acceptable (e.g., carbon_source: ["D-glucose", "D-galactose"])
2. **Property mappings**: Where to find each property in each dataset (e.g., "environmental_conditions.media.carbon_source")

## Key Features

- **Two-level filtering**:
  - Level 1: Repo/config level - excludes entire datasets that don't match
  - Level 2: Field level - returns matching field values within datasets
- **Three output modes**:
  - `conditions`: Just matching field values (no data retrieval)
  - `samples`: Sample-level metadata (one row per sample_id)
  - `full_data`: All measurements for matching samples
- **External configuration**: No hardcoded field specifications in codebase
- **Automatic path resolution**: Handles different nesting levels (repo vs field level)

## Setup

In [51]:
from pathlib import Path
import yaml
import pandas as pd
from tfbpapi.datainfo.filter_resolver import DatasetFilterResolver

## Example 1: Basic Filtering by Carbon Source

Let's start with a simple example: finding all samples grown on D-glucose.

## Understanding Field-Level vs Repo-Level Properties

A key challenge with heterogeneous datasets is that experimental conditions can be located in different places:

**Field-Level Definitions** (e.g., Harbison 2004):
- Properties are defined within a specific field's `definitions`
- Example: The `condition` field has definitions for "YPD", "HEAT", etc.
- Each definition contains `media`, `temperature_celsius`, etc.
- **Use the `field` parameter** to specify which field contains the definitions

**Repo/Config-Level Properties** (e.g., Kemmeren 2014):
- Properties are defined at the top level of the datacard
- All samples in the dataset share the same conditions
- Found in `experimental_conditions` at repo or config level
- **No `field` parameter needed** - just specify the path

Let's examine actual datacards to see these differences:

### Configuration Implications

Based on the actual datacard structures above:

**For Harbison 2004** (field-level):
```yaml
"BrentLab/harbison_2004":
  datasets:
    harbison_2004:
      carbon_source:
        field: condition       # Must specify which field
        path: media.carbon_source  # Path within each field value's definition
```

**For Kemmeren 2014** (repo-level):
```yaml
"BrentLab/kemmeren_2014":
  repo_level:
    carbon_source:
      path: environmental_conditions.media.carbon_source  # No 'field' needed
```

### When to Use Which Format

**Use `field` specification when**:
- The property varies between samples (different values in the dataset)
- The property is defined in a field's `definitions` section
- You need to filter to specific field values (e.g., only "YPD" condition)
- Example: Harbison 2004 where each condition has its own media/temperature

**Use repo-level (no `field`) when**:
- The property is the same for all samples in the dataset
- The property is in the top-level `experimental_conditions`
- You're filtering entire datasets in/out based on shared properties
- Example: Kemmeren 2014 where all samples share the same growth conditions

In [52]:
# Load Kemmeren 2014 (repo-level example)
from tfbpapi.datainfo import DataCard

kemmeren_card = DataCard("BrentLab/kemmeren_2014")

print("=" * 80)
print("KEMMEREN 2014 - Repo-Level Experimental Conditions")
print("=" * 80)

# Get repo-level experimental conditions (returns a dict)
exp_conds = kemmeren_card.get_experimental_conditions("kemmeren_2014")
print(f"\nRepo-level experimental_conditions exists: {bool(exp_conds)}")

if exp_conds:
    # exp_conds is a dict, not a Pydantic model
    print(f"\nTemperature: {exp_conds.get('temperature_celsius')}°C")
    
    media = exp_conds.get('media', {})
    print(f"Media name: {media.get('name')}")
    
    carbon_source = media.get('carbon_source', [])
    if carbon_source:
        compounds = [cs.get('compound') for cs in carbon_source]
        concentrations = [cs.get('concentration_percent') for cs in carbon_source]
        print(f"Carbon source: {compounds}")
        print(f"Concentrations: {concentrations}%")

# Check if there are field-level definitions
config = kemmeren_card.get_config("kemmeren_2014")
has_field_defs = False
for feature in config.dataset_info.features:
    if feature.definitions:
        has_field_defs = True
        print(f"\nField '{feature.name}' has definitions: {list(feature.definitions.keys())}")

if not has_field_defs:
    print("\nNo field-level definitions found")

print("\n" + "=" * 80)
print("Key Observation: Properties are at repo level, shared by all samples")
print("=" * 80)

KEMMEREN 2014 - Repo-Level Experimental Conditions

Repo-level experimental_conditions exists: True

Temperature: 30°C
Media name: synthetic_complete
Carbon source: ['D-glucose']
Concentrations: [2]%

No field-level definitions found

Key Observation: Properties are at repo level, shared by all samples


In [53]:
from tfbpapi.datainfo import DataCard
import json

# Load Harbison 2004 (field-level example)
harbison_card = DataCard("BrentLab/harbison_2004")

print("=" * 80)
print("HARBISON 2004 - Field-Level Definitions")
print("=" * 80)

# Check if there are repo-level experimental conditions
exp_conds = harbison_card.get_experimental_conditions("harbison_2004")
print(f"\nRepo-level experimental_conditions: {exp_conds}")

# Get field definitions for the 'condition' field
condition_defs = harbison_card.get_field_definitions("harbison_2004", "condition")
print(f"\nField 'condition' has {len(condition_defs)} definitions")
print(f"Condition values: {list(condition_defs.keys())}")

# Show a few example definitions
print("\nExample definitions:")
for cond_name in ["YPD", "HEAT", "GAL"]:
    if cond_name in condition_defs:
        definition = condition_defs[cond_name]
        print(f"\n{cond_name}:")
        print(f"  temperature_celsius: {definition.get('temperature_celsius')}")
        media = definition.get('media', {})
        print(f"  media.name: {media.get('name')}")
        carbon_source = media.get('carbon_source', [])
        if carbon_source:
            compounds = [cs.get('compound') for cs in carbon_source]
            print(f"  media.carbon_source: {compounds}")

print("\n" + "=" * 80)
print("Key Observation: Properties are in field definitions, NOT at repo level")
print("=" * 80)

HARBISON 2004 - Field-Level Definitions

Repo-level experimental_conditions: {}

Field 'condition' has 14 definitions
Condition values: ['YPD', 'SM', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90', 'Thi-', 'GAL', 'HEAT', 'Pi-', 'RAFF']

Example definitions:

YPD:
  temperature_celsius: 30
  media.name: YPD
  media.carbon_source: ['D-glucose']

HEAT:
  temperature_celsius: None
  media.name: YPD
  media.carbon_source: ['D-glucose']

GAL:
  temperature_celsius: 30
  media.name: yeast_extract_peptone
  media.carbon_source: ['D-galactose']

Key Observation: Properties are in field definitions, NOT at repo level


In [54]:
# Create a filter configuration
glucose_config = {
    "filters": {
        "carbon_source": ["D-glucose"]
    },
    "dataset_mappings": {
        "BrentLab/harbison_2004": {
            "datasets": {
                "harbison_2004": {
                    "carbon_source": {
                        "field": "condition",
                        "path": "media.carbon_source"
                    }
                }
            }
        }
    }
}

# Save to temporary file
config_path = Path("/tmp/glucose_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(glucose_config, f)

print("Configuration:")
print(yaml.dump(glucose_config, default_flow_style=False))


Configuration:
dataset_mappings:
  BrentLab/harbison_2004:
    datasets:
      harbison_2004:
        carbon_source:
          field: condition
          path: media.carbon_source
filters:
  carbon_source:
  - D-glucose



### Mode 0: Conditions Only

Get just the matching field values without retrieving any data.

In [55]:
# Create resolver
resolver = DatasetFilterResolver(config_path)

# Get matching conditions
results = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="conditions"
)

print("\nResults:")
for repo_id, result in results.items():
    print(f"\n{repo_id}:")
    if result["included"]:
        print(f"  Included: True")
        print(f"  Matching conditions:")
        for field, values in result["matching_field_values"].items():
            print(f"    {field}: {values}")
    else:
        print(f"  Included: False")
        print(f"  Reason: {result['reason']}")


Results:

BrentLab/harbison_2004:
  Included: True
  Matching conditions:
    condition: ['YPD', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90', 'HEAT']


### Understanding the Results

The `matching_field_values` shows which values of the `condition` field have D-glucose:
- YPD: Rich media with 2% glucose
- HEAT: Heat stress with glucose
- H2O2Hi/H2O2Lo: Oxidative stress with glucose
- Acid: Acidic stress with glucose
- Alpha: Alpha factor arrest with glucose
- BUT14/BUT90: Butanol treatment with glucose

Notice that GAL (galactose media) is NOT in the list - it was correctly filtered out.

## Example 2: Multiple Filter Criteria

Let's filter for samples with both D-glucose AND 30C temperature.

In [56]:
# Config with multiple filters
multi_filter_config = {
    "filters": {
        "carbon_source": ["D-glucose"],
        "temperature_celsius": [30]
    },
    "dataset_mappings": {
        "BrentLab/harbison_2004": {
            "datasets": {
                "harbison_2004": {
                    "carbon_source": {
                        "field": "condition",
                        "path": "media.carbon_source"
                    },
                    "temperature_celsius": {
                        "field": "condition",
                        "path": "temperature_celsius"
                    }
                }
            }
        }
    }
}

config_path = Path("/tmp/multi_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(multi_filter_config, f)

resolver = DatasetFilterResolver(config_path)
results = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="conditions"
)

print("Conditions with D-glucose AND 30C:")
matching = results["BrentLab/harbison_2004"]["matching_field_values"]["condition"]
print(matching)


Conditions with D-glucose AND 30C:
['YPD', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90']


Notice that HEAT is now excluded - it has glucose but is at 37C, not 30C. Multiple filter criteria use AND logic.

## Example 3: Mode 1 - Sample-Level Metadata

Retrieve actual sample metadata (one row per sample_id) for matching conditions.

In [57]:
# Use same config but change mode to 'samples'
results_samples = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="samples"
)

# Get the DataFrame
df_samples = results_samples["BrentLab/harbison_2004"]["data"]

print(f"Retrieved {len(df_samples)} samples")
print(f"\nDataFrame shape: {df_samples.shape}")
print(f"\nColumns: {list(df_samples.columns)}")
print(f"\nFirst few rows:")
df_samples.head()

Retrieved 304 samples

DataFrame shape: (304, 9)

Columns: ['sample_id', 'db_id', 'regulator_locus_tag', 'regulator_symbol', 'condition', 'target_locus_tag', 'target_symbol', 'effect', 'pvalue']

First few rows:


,sample_id,db_id,regulator_locus_tag,regulator_symbol,condition,target_locus_tag,target_symbol,effect,pvalue
0,1,0.0,YSC0017,MATA1,YPD,YAL023C,PMT2,0.749206,0.837071
1,2,1.0,YAL051W,OAF1,YPD,YAL047C,SPC72,1.966113,0.007623
2,3,2.0,YBL005W,PDR3,YPD,YBR001C,NTH2,0.974624,0.565974
3,4,3.0,YBL008W,HIR1,YPD,YBL037W,APL3,1.055576,0.358399
4,5,4.0,YBL021C,HAP3,YPD,YAL001C,TFC3,0.847317,0.811622


In [58]:
# Check the distribution of conditions in the retrieved samples
print("Condition distribution:")
print(df_samples['condition'].value_counts())

Condition distribution:
condition
YPD       204
H2O2Hi     39
H2O2Lo     28
RAPA       14
BUT14       8
Alpha       5
BUT90       4
Acid        2
Name: count, dtype: int64


## Example 4: Mode 2 - Full Data

Retrieve all measurements for matching samples (many rows per sample_id).

In [59]:
# Use same config but change mode to 'full_data'
results_full = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="full_data"
)

# Get the DataFrame
df_full = results_full["BrentLab/harbison_2004"]["data"]

print(f"Retrieved {len(df_full):,} rows (measurements)")
print(f"\nDataFrame shape: {df_full.shape}")
print(f"\nColumns: {list(df_full.columns)}")
print(f"\nFirst few rows:")
df_full.head()

Retrieved 1,892,704 rows (measurements)

DataFrame shape: (1892704, 9)

Columns: ['sample_id', 'db_id', 'regulator_locus_tag', 'regulator_symbol', 'condition', 'target_locus_tag', 'target_symbol', 'effect', 'pvalue']

First few rows:


,sample_id,db_id,regulator_locus_tag,regulator_symbol,condition,target_locus_tag,target_symbol,effect,pvalue
0,1,0.0,YSC0017,MATA1,YPD,YAL001C,TFC3,1.697754,0.068705
1,1,0.0,YSC0017,MATA1,YPD,YAL002W,VPS8,NaN,NaN
2,1,0.0,YSC0017,MATA1,YPD,YAL003W,EFB1,NaN,NaN
3,1,0.0,YSC0017,MATA1,YPD,YAL004W,YAL004W,0.745342,0.835929
4,1,0.0,YSC0017,MATA1,YPD,YAL005C,SSA1,NaN,NaN


In [60]:
# Compare to samples mode
n_samples = df_samples['sample_id'].nunique() if 'sample_id' in df_samples.columns else len(df_samples)
n_measurements = len(df_full)
measurements_per_sample = n_measurements / n_samples if n_samples > 0 else 0

print(f"Sample-level metadata: {n_samples} samples")
print(f"Full data: {n_measurements:,} measurements")
print(f"Average measurements per sample: {measurements_per_sample:,.0f}")

Sample-level metadata: 304 samples
Full data: 1,892,704 measurements
Average measurements per sample: 6,226


## Example 5: Filtering Across Multiple Datasets

The real power comes from filtering across multiple datasets with different structures.

In [61]:
# Config for multiple datasets
multi_dataset_config = {
    "filters": {
        "carbon_source": ["D-glucose", "D-galactose"]
    },
    "dataset_mappings": {
        "BrentLab/harbison_2004": {
            "datasets": {
                "harbison_2004": {
                    "carbon_source": {
                        "field": "condition",
                        "path": "media.carbon_source"
                    }
                }
            }
        },
        "BrentLab/kemmeren_2014": {
            "repo_level": {
                "carbon_source": {
                    "path": "environmental_conditions.media.carbon_source"
                }
            }
        }
    }
}

config_path = Path("/tmp/multi_dataset_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(multi_dataset_config, f)

resolver = DatasetFilterResolver(config_path)

# Filter multiple datasets
results = resolver.resolve_filters(
    repos=[
        ("BrentLab/harbison_2004", "harbison_2004"),
        ("BrentLab/kemmeren_2014", "kemmeren_2014")
    ],
    mode="conditions"
)

print("Results across multiple datasets:\n")
for repo_id, result in results.items():
    print(f"{repo_id}:")
    if result["included"]:
        print(f"  Included: True")
        if "matching_field_values" in result and result["matching_field_values"]:
            for field, values in result["matching_field_values"].items():
                print(f"  {field}: {values}")
        else:
            print("  No field-level conditions (all samples match at repo level)")
    else:
        print(f"  Included: False")
        print(f"  Reason: {result['reason']}")
    print()


Results across multiple datasets:

BrentLab/harbison_2004:
  Included: True
  condition: ['YPD', 'RAPA', 'H2O2Hi', 'H2O2Lo', 'Acid', 'Alpha', 'BUT14', 'BUT90', 'GAL', 'HEAT']

BrentLab/kemmeren_2014:
  Included: True
  No field-level conditions (all samples match at repo level)



## Example 6: Galactose-Only Filter

Let's filter for just galactose to see dataset exclusion in action.

In [62]:
galactose_config = {
    "filters": {
        "carbon_source": ["D-galactose"]
    },
    "dataset_mappings": {
        "BrentLab/harbison_2004": {
            "datasets": {
                "harbison_2004": {
                    "carbon_source": {
                        "field": "condition",
                        "path": "media.carbon_source"
                    }
                }
            }
        }
    }
}

config_path = Path("/tmp/galactose_filter.yaml")
with open(config_path, 'w') as f:
    yaml.dump(galactose_config, f)

resolver = DatasetFilterResolver(config_path)
results = resolver.resolve_filters(
    repos=[("BrentLab/harbison_2004", "harbison_2004")],
    mode="conditions"
)

print("Galactose-only filter results:")
result = results["BrentLab/harbison_2004"]
print(f"Included: {result['included']}")
if result['included']:
    matching = result["matching_field_values"]["condition"]
    print(f"Matching conditions: {matching}")


Galactose-only filter results:
Included: True
Matching conditions: ['GAL']


Only the GAL condition should match - it's the only one with galactose as the carbon source.

## How Path Resolution Works

The `DatasetFilterResolver` uses a two-phase approach to handle heterogeneous dataset structures:

### Phase 1: Property Location (Field vs Repo Level)

**Field-Level Resolution** (when `field` is specified):
```python
# Configuration
carbon_source:
  field: condition           # Look in the 'condition' field
  path: media.carbon_source  # Path within each field value's definition
```

The resolver:
1. Gets all definitions for the `condition` field
2. For each field value (YPD, HEAT, GAL, etc.), extracts the value at `media.carbon_source`
3. Returns field values where the extracted value matches the filter

**Repo-Level Resolution** (no `field` specified):
```python
# Configuration
carbon_source:
  path: environmental_conditions.media.carbon_source  # Top-level path
```

The resolver:
1. Looks at the top-level `experimental_conditions` in the datacard
2. Extracts the value at the specified path
3. Includes/excludes the entire dataset based on the match

### Phase 2: Path Navigation

Once the resolver knows WHERE to look (field vs repo level), it navigates to the property using dot notation.

**Automatic Fallback for Nested Paths**:

The resolver tries multiple path variants to handle different nesting depths:

1. **Try full path first**: `environmental_conditions.media.carbon_source`
2. **If not found, try without prefix**: `media.carbon_source`

This allows configs to work with both:
- Repo-level: `experimental_conditions.media.carbon_source` (finds it at step 1)
- Field-level: `media.carbon_source` (finds it at step 2)

**Example**: Harbison 2004 field definitions don't have an `environmental_conditions` wrapper, so:
- Path specified: `media.carbon_source`
- Directly finds `media` in the field definition
- No fallback needed

### Compound Extraction

For carbon source and similar properties, the resolver handles list-of-dict format:

```yaml
carbon_source:
  - compound: D-glucose
    concentration_percent: 2
  - compound: D-galactose  
    concentration_percent: 1
```

Automatically extracts to: `["D-glucose", "D-galactose"]`

### Why This Design?

This two-phase approach (field specification + path navigation) allows:

1. **No hardcoded field names** in the codebase
2. **Explicit configuration** - clear which field to examine  
3. **Flexibility** - works with any field name (`condition`, `treatment`, `strain`, etc.)
4. **Automatic nesting handling** - paths work at different hierarchy levels
5. **Heterogeneous datasets** - each dataset can specify its own structure

### Configuration Debugging Tips

If your filter isn't working:

1. **Check the datacard structure** - is the property at field or repo level?
2. **Verify the field name** - does the field exist? Is the name correct?
3. **Test the path** - use `DataCard.get_field_definitions()` to inspect
4. **Try simpler paths** - remove `environmental_conditions.` if not needed
5. **Use `conditions` mode first** - see which values match before retrieving data

## Practical Example: Creating a Filter Config File

Here's a complete example of a reusable filter configuration file.

In [63]:
# Create a comprehensive filter config
comprehensive_config = {
    "filters": {
        "carbon_source": ["D-glucose", "D-galactose", "D-raffinose"],
        "temperature_celsius": [30, 37]
    },
    "dataset_mappings": {
        "BrentLab/harbison_2004": {
            "datasets": {
                "harbison_2004": {
                    "carbon_source": {
                        "field": "condition",
                        "path": "media.carbon_source"
                    },
                    "temperature_celsius": {
                        "field": "condition",
                        "path": "temperature_celsius"
                    }
                }
            }
        },
        "BrentLab/kemmeren_2014": {
            "repo_level": {
                "carbon_source": {
                    "path": "environmental_conditions.media.carbon_source"
                },
                "temperature_celsius": {
                    "path": "environmental_conditions.temperature_celsius"
                }
            }
        }
    }
}

# Save to a persistent location
config_file = Path("my_filter_config.yaml")
with open(config_file, 'w') as f:
    yaml.dump(comprehensive_config, f)

print(f"Saved configuration to {config_file}")
print("\nConfiguration contents:")
print(yaml.dump(comprehensive_config, default_flow_style=False))


Saved configuration to my_filter_config.yaml

Configuration contents:
dataset_mappings:
  BrentLab/harbison_2004:
    datasets:
      harbison_2004:
        carbon_source:
          field: condition
          path: media.carbon_source
        temperature_celsius:
          field: condition
          path: temperature_celsius
  BrentLab/kemmeren_2014:
    repo_level:
      carbon_source:
        path: environmental_conditions.media.carbon_source
      temperature_celsius:
        path: environmental_conditions.temperature_celsius
filters:
  carbon_source:
  - D-glucose
  - D-galactose
  - D-raffinose
  temperature_celsius:
  - 30
  - 37



### Simplifying Single-Dataset Repositories

For repositories with a single dataset or when all datasets share the same structure, you can use `repo_level` for brevity:

```yaml
dataset_mappings:
  "BrentLab/kemmeren_2014":
    repo_level:
      carbon_source:
        path: environmental_conditions.media.carbon_source
      temperature_celsius:
        path: environmental_conditions.temperature_celsius
```

This applies the mappings to all datasets in the repo. For kemmeren_2014 (which has repo-level experimental_conditions), this is more concise than explicitly listing each dataset.

## Advanced: Hierarchical Configuration for Multi-Dataset Repositories

When a repository contains multiple datasets, you can use hierarchical configuration to:
1. Define repo-level mappings that apply to all datasets
2. Override or extend with dataset-specific mappings

This is especially useful when datasets share common properties but have some differences.

In [64]:
# Hierarchical configuration example
hierarchical_config = {
    "filters": {
        "carbon_source": ["D-glucose"],
        "temperature_celsius": [30]
    },
    "dataset_mappings": {
        # Repository with multiple datasets
        "BrentLab/multi_dataset_repo": {
            # Repo-level mappings apply to ALL datasets in this repo
            "repo_level": {
                "carbon_source": {
                    "path": "environmental_conditions.media.carbon_source"
                }
            },
            # Dataset-specific overrides
            "datasets": {
                "dataset1": {
                    # Inherits carbon_source from repo_level
                    # Adds temperature mapping for this specific dataset
                    "temperature_celsius": {
                        "path": "environmental_conditions.temperature_celsius"
                    }
                },
                "dataset2": {
                    # Inherits carbon_source from repo_level  
                    # Uses different temperature path
                    "temperature_celsius": {
                        "path": "custom.temp.location"
                    }
                },
                "dataset3": {
                    # Completely overrides carbon_source for this dataset
                    "carbon_source": {
                        "path": "media.carbon"  # Different structure
                    },
                    "temperature_celsius": {
                        "path": "temp"  # Top-level property
                    }
                }
            }
        }
    }
}

print("Hierarchical configuration:")
print(yaml.dump(hierarchical_config, default_flow_style=False))

Hierarchical configuration:
dataset_mappings:
  BrentLab/multi_dataset_repo:
    datasets:
      dataset1:
        temperature_celsius:
          path: environmental_conditions.temperature_celsius
      dataset2:
        temperature_celsius:
          path: custom.temp.location
      dataset3:
        carbon_source:
          path: media.carbon
        temperature_celsius:
          path: temp
    repo_level:
      carbon_source:
        path: environmental_conditions.media.carbon_source
filters:
  carbon_source:
  - D-glucose
  temperature_celsius:
  - 30



### Key Benefits of Hierarchical Configuration

1. **DRY Principle**: Define common mappings once at repo level
2. **Flexibility**: Override for specific datasets that differ
3. **Maintainability**: Update repo-level mappings to affect all datasets at once
4. **Clarity**: Explicitly shows which datasets inherit vs override

### How Merging Works

For each dataset, the resolver:
1. Starts with repo-level mappings
2. Overlays dataset-specific mappings (these take precedence)
3. Returns the merged mapping for that dataset

In the example above:
- `dataset1` gets `carbon_source` from repo_level + its own `temperature_celsius`
- `dataset2` gets `carbon_source` from repo_level + its own (different) `temperature_celsius`
- `dataset3` overrides both properties completely

## Summary

The `DatasetFilterResolver` provides:

1. **Flexible filtering** across heterogeneous datasets
2. **External configuration** - no code changes needed for new filters
3. **Two-level filtering** - dataset exclusion and field-level matching
4. **Three output modes** - from lightweight (conditions only) to full data
5. **Field-aware resolution** - explicit specification of where properties are located
6. **Automatic path resolution** - handles different nesting levels gracefully

### When to Use Each Mode

- **`conditions`**: Quick exploration - which datasets have matching samples? Which field values match?
- **`samples`**: Sample metadata analysis - how many samples match? What are their properties?
- **`full_data`**: Full analysis - retrieve all measurements for downstream processing

### Best Practices for Configuration

#### 1. Field Specification
- **Always specify `field`** when the property is in field-level definitions
- **Never specify `field`** when the property is at repo/config level
- When in doubt, check the datacard: if the property is in a field's `definitions`, use `field`

#### 2. Property Paths
- Use the simplest path that works (avoid unnecessary nesting)
- For field-level properties, typically: `media.carbon_source` (not `environmental_conditions.media.carbon_source`)
- For repo-level properties, include full path: `environmental_conditions.media.carbon_source`

#### 3. Hierarchical Configuration
- Use `repo_level` for properties shared across all datasets in a repository
- Use `datasets` to override or extend for specific datasets
- This keeps configuration DRY and maintainable

#### 4. Filter Development Workflow
1. **Start with `conditions` mode** to verify your filters work correctly
2. **Check the results** - do the field values make sense?
3. **Use `samples` mode** to understand sample counts before downloading full data
4. **Only use `full_data`** when you're ready to retrieve the actual measurements

#### 5. Configuration Management
- Create reusable YAML configs for common filter scenarios
- Document your filter configs with comments explaining the scientific rationale
- Version control your configs alongside your analysis code
- Test configs with `conditions` mode before using in production pipelines

#### 6. Debugging
- If a dataset shows `included: False`, check the `reason` field
- Use `DataCard.get_field_definitions()` to inspect field structure
- Use `DataCard.get_experimental_conditions()` to inspect repo-level conditions
- Start simple (one filter, one dataset) then expand

### Common Patterns

**Pattern 1: Single property, field-level**
```yaml
filters:
  carbon_source: ["D-glucose"]
dataset_mappings:
  "BrentLab/harbison_2004":
    datasets:
      harbison_2004:
        carbon_source:
          field: condition
          path: media.carbon_source
```

**Pattern 2: Multiple properties, repo-level**
```yaml
filters:
  carbon_source: ["D-glucose"]
  temperature_celsius: [30]
dataset_mappings:
  "BrentLab/kemmeren_2014":
    repo_level:
      carbon_source:
        path: environmental_conditions.media.carbon_source
      temperature_celsius:
        path: environmental_conditions.temperature_celsius
```

**Pattern 3: Mixed (repo-level + dataset overrides)**
```yaml
dataset_mappings:
  "BrentLab/multi_dataset_repo":
    repo_level:
      carbon_source:
        path: environmental_conditions.media.carbon_source
    datasets:
      dataset1:
        temperature_celsius:
          field: condition
          path: temperature_celsius
```

### Next Steps

- Explore the `example_filter_config.yaml` file for more examples
- Try filtering across multiple datasets with different structures
- Integrate with downstream analysis pipelines using the exported DataFrames
- Consider creating a library of reusable filter configs for your common use cases